# Exploring Sentiment Analysis

This notebook will be a guided exploration of sentiment analysis with Python. We will look at what sentiment analysis is, how it works using Python, and will explore different pathways for conducting the analysis, including the Gale Digital Scholar Lab sentiment analysis tool. This notebook also provides an opportunity to explore the differences between sentiment analysis lexicons, and to run a sentiment analysis on your own document set.

Don't worry if you've never coded before - this notebook will guide you through the process. If you have coded before, especially using Python, there may be some additional explanations that you can just skim, as they are likely review.

## Table of Contents
- Part 0: Introductions
    - [How to Use Jupyter Notebooks](#How-to-Use-Jupyter-Notebooks)
    - [General Instructions](#General-Instructions)
- [Part 1: What is Sentiment Analysis?](#Part-1:-What-is-Sentiment-Analysis?)
- [Part 2: Sentiment Analysis Using AFINN](#Part-2:-Sentiment-Analysis-Using-AFINN)
- [Part 3: Sentiment Analysis of a Larger Content Set Using AFINN](#Part-3:-Sentiment-Analysis-of-a-Larger-Content-Set-Using-AFINN)
- [Additional Exploration](#Additional-Exploration)

### How to Use Jupyter Notebooks
This is a Jupyter Notebook, which is a type of file that facilitates the process of running Python code. These notebooks are separated into blocks of either markdown text, like this block, or executable code, like several of the blocks below. There shouldn't be anything you need to do to a markdown block, but if you click in it and want to return to the formatted text, either click "ctrl enter" or use the run button (single triangle) at the top of the screen. To run the code in the code block, you can do the same thing - click inside the code block, then click "ctrl enter" or use the run button.

### General Instructions
This notebook will contain a few markdown blocks explaining the overarching idea of what we're doing, and then several code blocks. Each code block will have comments explaining each line of code and what they're doing, which you can read to better understand how the code is working. To run each code block, click inside it and use "ctrl enter," or click the run button and observe the output.

## Part 1: What is Sentiment Analysis?
Sentiment analysis is a process that attempts to classify the overall sentiment of a piece of text as positive, negative, or neutral. There are various ways to do this, but one of the most common is to use a lexicon (also called a valence dictionary), which is essentially a long list of words that are assigned a "score" to describe how positive or negative that word is. We can then compare the words in our text(s) to the words in the lexicon and use their associated scores to get an idea of the overall sentiment of our document.

## Part 2: Sentiment Analysis Using AFINN
AFINN is one of these lexicons, and is both popular and fairly simple. It is a list of over 3300 words, each assigned a score of -5 (most negative) to 5 (most positive), and was created from 2009 to 2011 by Finn Årup Nielsen. In the code blocks below, we'll explore how to use it.

In [ ]:
# This is a comment - a line (or lines) of text in a code block that won't interfere with the code
# A comment line starts with #

# The lines below are importing certain software packages we will need

# pandas - a library that makes it easier to work with lots of data, create nice tables, etc.
import pandas as pd 

# matplotlib - a library that lets us make graphs
import matplotlib.pyplot as plt 

# seaborn - a library that builds off of the previous library to make the graphs look nicer
import seaborn as sns 

# os - a library to work with files and folders
import os 

# a specific mathematical function for later
from math import ceil

# Commands that start with ! are similar to running that command in a terminal window
!pip install afinn
# afinn - the first lexicon (language model) we'll test
from afinn import Afinn 

# instantiating the tool
afn = Afinn()

After running the code above, you should have something that ends with "successfully installed afinn-0.1" or something similar.

This next code block is where the interesting things begin:

In [ ]:
# list of sentences to analyze
sentences = ['les gens pensent aux chiens', 'i hate flowers', 'he is kind and smart', 'we are kind to good people']

#----------------#
# Assign each sentence a score
#----------------#
# Creating an empty list to store the results
scores = []
# Go through each sentence in the sentences list
for sentence in sentences: 
    # Assigning a score for each sentence
    score = afn.score(sentence) 
    # Add the score of the sentence to the scores list
    scores.append(score) 

#----------------#
# Assign each sentence a sentiment label
#----------------#
# Creating an empty list to store the sentiment labels
labels = [] 
# Go through each score in the scores list
for score in scores: 
    # Assign 'positive' label if the score is above 1
    if score > 1:
        labels.append('positive') 
    # Assign 'negative' label if the score is below 1
    elif score < -1:
        labels.append('negative') 
    # Assign 'neutural' label otherwise
    else:
        labels.append('neutral') 

# Here we are just combining the scores and sentiments we calculated above to make a table
# First, we combine all three lists
table_data = list(zip(sentences, scores, labels))
# Next, we make the table with column titles
afinn_intro_df = pd.DataFrame(table_data, columns=['Sentence', 'Score', 'Label']) 
# Lastly, we display the table
afinn_intro_df

In the table above, we can see the basic principle of what's happening - our code is running through the list of sentences we provide, assigning each one a score, and then labeling it as positive, negative, or neutral based on that score (and whether that score is above 1, below -1, or in the middle).

One question this might raise, though, is how those score are being determined. For example, take a look at the last two lines of the table. Why does 'he is kind and smart' get a score of 3.0, while 'we are kind to good people' get a score of 5.0? We might assume that words like 'we', 'and', 'people', etc. have no sentiment, and that the words affecting the scores would be 'kind' and 'smart' for the first sentence, and 'kind' and 'good' for the second - but why the difference in scores? Is 'good' really 2 points more positive than 'smart'? Who decided that, and how? What about the difference between sentences 1 and 2 - why is the sentence with two fairly positive words ('kind' and 'smart') as positive as the sentence with one negative word ('hate') is negative (+3 vs -3)? Is that an accurate assessment of the sentences' sentiment? Could there be alternatives?

## Part 3: Sentiment Analysis of a Larger Content Set Using AFINN

### Running AFINN.score on each document

To explore these ideas more, let's test AFINN with a more robust dataset and longer documents. For this demo, we're going to be using a modified version of the Roberto Calvi dataset from Gale Digital Scholar Lab comprising 210 documents, but later you'll have the opportunity to try it on your own content set, if you so wish. This dataset was cleaned with Gale's default cleaning configuration.

First, we need to actually find the documents we want to look at, which is what the code below does. (There won't be any output from this code block, but you still need to run it!)

In [ ]:
# The name of the folder we create to contain our documents
directory = 'roberto calvi data' 

# Create an empty list to track documents with
documents = [] 
# Goes into the folder with the name we gave above, and go through each filename it finds
for doc in os.scandir(directory): 
    # Ignores hidden files like '.ipynb_checkpoints' that are created by JupyterHub
    if not doc.name.startswith('.'):
        # Adds that name to the list
        documents.append(doc.name) 

After we've done that, we can see what happens if we use the same `afn.score()` function we used in Part 2 on our documents:

In [ ]:
# Creating an empty list to track the sentiment scores of each document
scores = []

# go through each document in the list we made in the last code block
for doc in documents: 
    # creates a 'path' that tells the code which folder and file to look at
    filename = str(directory + '/' + doc) 
    
    # opens that document
    with open(filename) as f:
        # gets all of the text from the file and brings it over so we can work with it
        contents = f.read() 
        # scores the file's text using AFINN's lexicon
        score = afn.score(contents)
        # adds that score to the list so we can make it a column in the table
        scores.append(score) 

# same labeling process as we did with our first experiment with AFINN 
# - note the thresholds for neutrality! Why might we have changed them?
labels = []
for score in scores:
    if score > 3:
        labels.append('positive')
    elif score < -3:
        labels.append('negative')
    else:
        labels.append('neutral')

# creating the table using the same process as we did with AFINN:
table_data = list(zip(documents, scores, labels)) 
afinn_total_df = pd.DataFrame(table_data, columns=['File', 'Score', 'Label']) 
afinn_total_df

Above, we get a similar table to the one we made in Part 2, except now it's 210 rows, and instead of sentences, we have file names (those aren't what was actually scored though - they just stand in for the actual text because it would be too long to read otherwise, but if you want to see what the original text was, you can do so by opening the 'roberto calvi data' folder from the file pane and opening a given file).

While this table is great, pandas (the library we're using to make the tables) shows only 10 rows of longer tables (the first and last 5). To get a better understanding of our results, we have some different options.

In [ ]:
# because pandas only shows the first and last 5 rows of a table
# the code below helps us get a better understanding of the distribution of sentiment labels

print("Sentiment Label Counts")
# get a list of unique values for that column and the number of times each appeared
print(afinn_total_df['Label'].value_counts()) 

The above print-out shows us the distribution of our labels - we can see that in this content set, 172 documents were labeled 'negative', 31 were labeled 'positive', and 7 were labeled 'neutral'.

It might also be helpful to visualize the distribution in a graph. There's many ways we could do that, but two (a swarmplot, or a categorical scatterplot with points adjusted vertically to not overlap and better show the distribution, and a boxplot, which shows the median, 25th percentile, 75th percentile, minimum value, and maxiumum value) are shown below:

In [ ]:
# increasing the text size so it's easier to read
sns.set(font_scale=1.3)

# sizes the graphs nicely and creating the two spaces to graph in
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10)) 
# making the swarmplot
sns.swarmplot(ax=ax1, x="Score", y="Label", hue="Label", palette="magma", data=afinn_total_df) 
# a swarmplot is a categorical scatterplot with points adjusted to not overlap 
# so it's easier to see the distribution

# adding a title to the graph
ax1.set_title('AFINN Scores for Roberto Calvi Content Set, Default') 

# making the boxplot
sns.boxplot(ax=ax2, x=afinn_total_df["Score"]);
# The ; is used to remove weird '<Axes.subplot> text

From the graphs above, we can get a good grasp of not just how many of each label type there were, but also what the distribution of scores is (how many are negative? What's the most negative score, or the most positive score? What is the average score or so? etc.). From them, we can see that the average score is a little under 0, which aligns with what we'd expect from having 172 documents labeled "negative" compared to 31 positive ones and 7 neutral ones.

One thing that is interesting to note is the actual scores themselves. They seem to spread between -150 and 100, which might immediately seem strange to you, especially when compared to the sentiment analysis tool in Gale Digital Scholar Lab, which we know provides document scores between -5 and 5.

One avenue for exploration is to compare the scores calculated for a given document here with the score assigned in the Lab. To make that easier, we've created a "search" tool, below, to find the score of a document when you know the title. You can edit the `search_term` variable in the code block below and re-run it as many times as you like to find documents that match your search term. You could also go into the Lab and find a document from the Roberto Calvi dataset in the Sentiment Analysis tool, and search for that document here, in order to compare the scores.

In [ ]:
# tool to look at any rows that contain search term in file name, case insensitive, sorted alphabetically
# note that spaces do not work - words must be separated by underscores (_) as they would be in the file names

search_term = 'calvi' # change the text within the ''s and then run code block to search for a different term

afinn_total_df[afinn_total_df['File'] # Search in this column
               .str # Making sure the values in the column are strings
               .contains(search_term, case=False) # Check for the search term, ignoring case-sensitivity
              ].sort_values('File')# Then sort the output

<hr>

##  Additional Exploration
### Sentiment Analysis with VADER (Valence Aware Dictionary and sEntiment Reasoner)

VADER is another way of classifying the sentiment of texts. Unlike AFINN, VADER tells users how positive, negative, and neutral it feels the text is, rather than just the sum of all the scored tokens in the text. We explore what it looks like more in the code below.

In [ ]:
# This code installs Vader elements we need to use later

!pip install vaderSentiment
import vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

When the code runs successfully, you should see something similar to "successfully installed vaderSentiment".

In [ ]:
# instantiating the tool
analyser = SentimentIntensityAnalyzer()

# the same list of sentences as we used for AFINN
sentences = ['les gens pensent aux chiens', 'i hate flowers', 'he is kind and smart', 'we are kind to good people']

# empty list to track the scores we get back
scores = [] 
# go through each sentence
for sentence in sentences:
    # track the scores for each sentence
    scores.append(analyser.polarity_scores(sentence)) 

"""
VADER's scoring function creates what Python calls a 'dictionary' - essentially a set of keys and values associated with
those keys. This means that we don't just get a single score - instead, we get a set of scores that includes the negative
score, positive score, neutral score, and compound (overall) score for a single document. To get those all into rows of a table
for easier viewing, we go through each dictionary in the 'scores' list and make a list of all the negative scores,
then of all the neutral scores, etc., which is the code below.
"""
neg_list = []
neu_list = []
pos_list = []
compound_list = []
for d in scores:
    neg_list.append(d['neg'])
    neu_list.append(d['neu'])
    pos_list.append(d['pos'])
    compound_list.append(d['compound'])

# assign label based on VADER's compound score - note the threshold of 0.1 
# - how did we decide that? What if we change it?
labels = []
for score in compound_list:
    if score > 0.1:
        labels.append('positive')
    elif score < -0.1:
        labels.append('negative')
    else:
        labels.append('neutral')

# making and displaying the table
vader_data = list(zip(sentences, neg_list, neu_list, pos_list, compound_list, labels)) 
vader_intro_df = pd.DataFrame(vader_data, columns=['Sentence', 'Percent Negative', 'Percent Neutral', 
                                                   'Percent Positive','Compund Score', 'Label']) 
vader_intro_df 

Above, we can see the results VADER gives when looking at the same original 4 sentences we used when introducing AFINN. Just for fun, let's also look at what AFINN's table for those sentences looked like:

In [ ]:
afinn_intro_df 
# this is the table we made all the way back in part 2 - displaying it here for easier comparison

What do you notice about the differences between AFINN and VADER? Is one or the other more easily understandable to you, and if so, why?

### VADER with a Larger Content Set
To get a better sense of VADER, let's also look at it with a larger content set. This uses the same collection of documents we used with AFINN, above.

In [ ]:
# instantiating the tool
analyser = SentimentIntensityAnalyzer() 

# a list to keep track of the documents
scores = [] 

# go through each document in the list we made when working with AFINN
for doc in documents: 
    # appending the document name to the directory name so we can find the file
    filename = str(directory + '/' + doc) 
    
    # lets us access the actual text inside the file
    with open(filename) as f:
        # gets all of the text from the file and brings it over so we can work with it
        contents = f.read()
        # scores the file's text
        score_dict = analyser.polarity_scores(contents)
        # adds that score to the list so we use the scores later
        scores.append(score_dict) 

# same thing as above - extracting each score type into a list
neg_list = []
neu_list = []
pos_list = []
compound_list = []
for d in scores:
    neg_list.append(d['neg'])
    neu_list.append(d['neu'])
    pos_list.append(d['pos'])
    compound_list.append(d['compound'])

# labeling the documents based on their compound score 
# - note the threshold for what is neutral and not
labels = []
for score in compound_list:
    if score > 0.1:
        labels.append('positive')
    elif score < -0.1:
        labels.append('negative')
    else:
        labels.append('neutral')

# making and displaying the table
table_data = list(zip(documents, neg_list, neu_list, pos_list, compound_list, labels)) 
vader_df = pd.DataFrame(table_data, columns=['File', 'Percent Negative', 'Percent Neutral', 'Percent Positive',
                                             'Overall Score', 'Label'])
vader_df

What do you notice about it?

Lastly, we can look at the counts of each label to see the distribution of labels, as well as create graphs to visualize the distribution. We can then compare the results with the label counts we got when running AFINN.

In [ ]:
# because pandas only shows the first and last 5 rows of a table
# the code below helps us get a better understanding of the distribution of sentiment labels

# VADER's scores
print("Sentiment Label Counts - VADER")
print(vader_df['Label'].value_counts()) 

# Creating a separator between the tools
print('-----')

# AFINN's scores
print("Sentiment Label Counts - AFINN")
print(afinn_total_df['Label'].value_counts()) 

In [ ]:
# Creating the graphs to represent the data
sns.set(font_scale=1.3)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 15)) 

sns.swarmplot(ax=ax1, x="Overall Score", y="Label", hue="Label", palette="magma", data=vader_df);
ax1.set_title('VADER Scores for Your Content Set') 

# a striplot is a categorical scatterplot with jitter added to reduce overlap 
# so it's easier to see the distribution
sns.stripplot(ax=ax2, x="Overall Score", y="Label", hue="Label", palette="magma", data=vader_df);

# making the boxplot
sns.boxplot(ax=ax3, x=vader_df["Overall Score"]);

What do you note about the differences between AFINN and VADER? Why do you think those differences might be? Do you prefer one of the two, and why or why not?

#### A Quick Note About Visualizations

In the last code block, you may have seen a red warning box appear above your graphs. You may also have noticed an additional graph - a stripplot - has been added as well. Why?

Well, if you did get the red warning box, you can see that it's warning us that some percentage of our points couldn't be placed on the swarmplot (the categorical scatterplot we'd been using previously) and to either reduce our point size or to use a stripplot instead. We did do that - which is why the stripplot is also included. We also wanted to keep the swarmplot, though, and add this note to show how different data can look on two different graphs as another example of how a seemingly small change can have a big difference in your analysis. What do you notice about the difference between the two graphs? Do you find one easier to read or easier to understand? Why do you think that might be?